In [1]:
import torch
import pandas as pd
import numpy as np
import regex as re
from epsilon_transformers.persistence import Persister


/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

In [91]:
persister = Persister(save_dir="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000/")
# model_path="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/checkpoint_801_tokens_40000000.pt"
#model=persister.load_model(model_path)
model = persister.load_final_model()

[Persister] Found 4007 checkpoints in /Users/sbhandari/Documents/GitHub/epsilon-transformers/models/1layer_0.15_0.6_lr0.02_ckpt4000/epsilon-transformers/models/linear_mess3/1layer_0.15_0.6_lr0.02_ckpt4000


In [3]:
from epsilon_transformers.process.processes import PROCESS_REGISTRY
from torch import device
process_name = 'Linear_Mess3'
process_params ={
    "x": 0.15,
    "a": 0.6
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)
history=process.generate_process_history(total_length=10)
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)    

In [4]:
def measure_residual_norms(model, input_seq,layer_idx=None):
    def filter(name):
        return any(name.endswith(suffix) for suffix in ['out','_normalized','_pre','_post','_mid'])
    with torch.no_grad():
        _,cache=model.run_with_cache(input_seq,names_filter=filter)
    norms=[]
    for hook_name, act in cache.items():
        if act.dim()>=2:
            norm_val=act.flatten(start_dim=2).norm(dim=-1).mean().item()
        else:
            norm_val=act.norm().item()
        match=re.search(r'blocks\.(\d+)\.', hook_name)
        layer_idx=int(match.group(1)) if match else -1
        component=hook_name.split('.')[-1]
        norms.append({
            "layer_idx": layer_idx,
            "hook_name": hook_name,
            "component": component,
            "norm": norm_val  })
    return pd.DataFrame(norms)

In [ ]:
import torch
import numpy as np
from sklearn.linear_model import LinearRegression

def analyze_mlp_math(model, process, layer_idx=0, num_seqs=128, seq_len=10, device=None):
    
    if device is None:
        if torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if hasattr(process, "generate_batch_gpu"):
        input_batch = process.generate_batch_gpu(
            batch_size=num_seqs, seq_len=seq_len, device=device
        )
    # else:
    #     seqs = [
    #         process.generate_process_history(total_length=seq_len).symbols
    #         for _ in range(num_seqs)
    #     ]
    #     input_batch = torch.tensor(seqs, dtype=torch.long, device=device)

    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized",
        f"blocks.{layer_idx}.hook_mlp_out",
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_batch, names_filter=cache_names)

    #[B, T, d_model]
    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    tokens = input_batch.cpu().numpy()  # [B, T]

    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    results = {}

    for token_id in range(3):  # tokens 0,1,2
        mask = (tokens_flat == token_id)
        X = mlp_in_flat[mask]
        Y = mlp_out_flat[mask]

        if X.shape[0] < D: 
            print(f"Token {token_id}: not enough samples ({X.shape[0]}), skipping.")
            continue

        # Fit linear map Y ≈ X @ M  (no bias; LN should remove mean shifts)
        reg = LinearRegression(fit_intercept=False).fit(X, Y)
        M_learned = reg.coef_        # [D, D]
        r2 = reg.score(X, Y)

        results[token_id] = {"M": M_learned, "R2": r2}
        print(f"Token {token_id}: MLP linear fit R^2 = {r2:.4f}")

    return results

In [6]:
res = analyze_mlp_math(model, process, layer_idx=0, num_seqs=256, seq_len=10)

Token 0: MLP linear fit R^2 = 0.9708
Token 1: MLP linear fit R^2 = 0.9740
Token 2: MLP linear fit R^2 = 0.9631


In [ ]:
def run_activation_to_beliefs_regression(activations, ground_truth_beliefs):

    assert activations.shape[0] == ground_truth_beliefs.shape[0]
    assert activations.shape[1] == ground_truth_beliefs.shape[1]

    batch_size, n_ctx, d_model = activations.shape
    belief_dim = ground_truth_beliefs.shape[-1]
    activations_flattened = activations.view(-1, d_model) # [batch * n_ctx, d_model]
    ground_truth_beliefs_flattened = ground_truth_beliefs.view(-1, belief_dim) # [batch * n_ctx, belief_dim]
    
    regression = LinearRegression()
    regression.fit(activations_flattened, ground_truth_beliefs_flattened)

    belief_predictions = regression.predict(activations_flattened) # [batch * n_ctx, belief_dim]
    belief_predictions = belief_predictions.reshape(batch_size, n_ctx, belief_dim)

    return regression, belief_predictions



In [ ]:
def get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10):
    """
    Generates NEW random data, grabs MLP inputs/outputs, and uses the 
    PRE-TRAINED regression probe to estimate beliefs for them.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.backends.mps.is_available(): device = torch.device("mps")

    # Generate random sequences)
    if hasattr(process, "_ensure_gpu_tensors"):
        process._ensure_gpu_tensors(device)
    # elif hasattr(process, "transition_matrix"):
    #      if not isinstance(process.transition_matrix, torch.Tensor):
    #          process.transition_matrix = torch.tensor(process.transition_matrix, dtype=torch.float32)
    #      process.transition_matrix = process.transition_matrix.to(device)

    if hasattr(process, "generate_batch_gpu"):
        inputs = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    else:
        seqs = [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)]
        inputs = torch.tensor(seqs, dtype=torch.long, device=device)
    
    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized", # MLP In
        f"blocks.{layer_idx}.hook_mlp_out",         # MLP Out
        f"blocks.{layer_idx}.hook_resid_mid",       # For Belief Projection
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(inputs, names_filter=cache_names)

    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    resid_for_probe = cache[cache_names[2]].cpu().numpy()
    tokens = inputs.cpu().numpy()

    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    resid_flat = resid_for_probe.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    beliefs_flat = regression.predict(resid_flat) # [N, 3]

    return mlp_in_flat, mlp_out_flat, tokens_flat, beliefs_flat


In [ ]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np

def analyze_per_region_linearity(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- 1. Per-Region Linearity Analysis ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        
        if len(X) < 50: continue
            
        print(f"Token {token_id}: Global R^2 = {LinearRegression(fit_intercept=False).fit(X, Y).score(X, Y):.4f}")
        
        # Cluster
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        for i in range(n_clusters):
            c_mask = (labels == i)
            if c_mask.sum() < 20: continue
            r2 = LinearRegression(fit_intercept=True).fit(X[c_mask], Y[c_mask]).score(X[c_mask], Y[c_mask])
            print(f"  Cluster {i} (n={c_mask.sum()}): R^2 = {r2:.4f}")



In [10]:
from epsilon_transformers.analysis.activation_analysis import get_beliefs_for_transformer_inputs
mixed_state_tree = process.derive_mixed_state_presentation(depth=10 + 1)
MSP_transition_matrix = mixed_state_tree.build_msp_transition_matrix()

# in order to plot the belief states in the simplex, we need to get the paths and beliefs from the MSP
tree_paths, tree_beliefs = mixed_state_tree.paths_and_belief_states
msp_beliefs = [tuple(round(b, 5) for b in belief) for belief in tree_beliefs]
print(f"Number of Unique beliefs: {len(set(msp_beliefs))} out of {len(msp_beliefs)}")

# now lets index each belief
msp_belief_index = {b: i for i, b in enumerate(set(msp_beliefs))}
device = 'cpu'
train_config = persister.load_training_config()
transformer_inputs = [x for x in tree_paths if len(x) == 10]
transformer_inputs = torch.tensor(transformer_inputs, dtype=torch.int).to(device)

# print first few batches
print(transformer_inputs[:5])


transformer_input_beliefs, transformer_input_belief_indices = get_beliefs_for_transformer_inputs(transformer_inputs, msp_belief_index, tree_paths, tree_beliefs)
print(f"Transformer Input Beliefs: {transformer_input_beliefs.shape}, Transformer Input Belief Indices: {transformer_input_belief_indices.shape}")

_, activations = model.run_with_cache(transformer_inputs, names_filter=lambda x: 'resid_mid' in x)
#_, activations = model.run_with_cache(transformer_inputs)
#activations['blocks.0.hook_resid_mid'].shape  'ln_final.hook_normalized'
#activations = activations['blocks.3.hook_resid_post']
activations.keys()
print(activations.keys())
#acts = torch.concatenate((activations["blocks.0.ln1.hook_normalized"], activations["blocks.1.ln1.hook_normalized"], activations["blocks.2.ln1.hook_normalized"], activations["blocks.3.ln1.hook_normalized"]), dim=-1)
#acts = activations['ln_final.hook_normalized']
acts = activations['blocks.0.hook_resid_mid']
regression_mid, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

Number of Unique beliefs: 265720 out of 265720
tensor([[0, 1, 0, 0, 0, 2, 2, 2, 2, 0],
        [0, 1, 0, 0, 0, 2, 2, 2, 2, 1],
        [0, 1, 0, 0, 0, 2, 2, 2, 2, 2],
        [1, 2, 0, 2, 0, 2, 2, 0, 0, 0],
        [1, 0, 0, 0, 2, 1, 0, 1, 1, 0]], dtype=torch.int32)
Transformer Input Beliefs: torch.Size([59049, 10, 3]), Transformer Input Belief Indices: torch.Size([59049, 10])
dict_keys(['blocks.0.hook_resid_mid'])
(59049, 10, 3)


In [11]:
_,activations = model.run_with_cache(transformer_inputs, names_filter=lambda x: 'resid_post' in x)
curr_resid_post = activations['blocks.0.hook_resid_post']
print(curr_resid_post.shape)
print(transformer_input_beliefs.shape)
regression_post, belief_predictions_post = run_activation_to_beliefs_regression(curr_resid_post, transformer_input_beliefs)

torch.Size([59049, 10, 64])
torch.Size([59049, 10, 3])


In [12]:
regression, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

(59049, 10, 3)


In [13]:
analyze_per_region_linearity(model, process, regression,n_clusters=10)


--- 1. Per-Region Linearity Analysis ---
Token 0: Global R^2 = 0.9694
  Cluster 0 (n=80): R^2 = 0.9708
  Cluster 1 (n=91): R^2 = 0.9650
  Cluster 2 (n=62): R^2 = 0.9878
  Cluster 3 (n=89): R^2 = 0.9929
  Cluster 4 (n=114): R^2 = 0.9903
  Cluster 5 (n=71): R^2 = 1.0000
  Cluster 6 (n=62): R^2 = 0.9859
  Cluster 7 (n=102): R^2 = 0.9943
  Cluster 8 (n=119): R^2 = 0.9926
  Cluster 9 (n=87): R^2 = 0.9543
Token 1: Global R^2 = 0.9747
  Cluster 0 (n=54): R^2 = 1.0000
  Cluster 1 (n=81): R^2 = 0.9599
  Cluster 2 (n=82): R^2 = 0.9768
  Cluster 3 (n=47): R^2 = 0.9850
  Cluster 4 (n=89): R^2 = 0.9937
  Cluster 5 (n=66): R^2 = 0.9727
  Cluster 6 (n=82): R^2 = 0.9643
  Cluster 7 (n=96): R^2 = 1.0000
  Cluster 8 (n=96): R^2 = 0.9924
  Cluster 9 (n=141): R^2 = 0.9897
Token 2: Global R^2 = 0.9631
  Cluster 0 (n=56): R^2 = 0.9861
  Cluster 1 (n=85): R^2 = 0.9850
  Cluster 2 (n=70): R^2 = 0.9633
  Cluster 3 (n=69): R^2 = 0.9642
  Cluster 4 (n=89): R^2 = 1.0000
  Cluster 5 (n=77): R^2 = 0.9739
  Cluster

In [17]:
def inspect_variance(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- Variance Inspection ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs, seq_len)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        if len(X) < 50: continue

        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        print(f"\nToken {token_id}:")
        for i in range(n_clusters):
            c_mask = (labels == i)
            Y_c = Y[c_mask]
            if len(Y_c) < 10: continue
            
            # Std Dev of Output Norms
            y_std = Y_c.std(axis=0).mean()
            y_norm = np.linalg.norm(Y_c, axis=1).mean()
            
            # Linearity check again
            reg = LinearRegression(fit_intercept=True).fit(X[c_mask], Y_c)
            r2 = reg.score(X[c_mask], Y_c)
            
            print(f"  Cluster {i}: R2={r2:.4f} | Y_std={y_std:.4f} | Y_norm={y_norm:.4f}")
inspect_variance(model, process, regression, layer_idx=0, num_seqs=512, seq_len=10,n_clusters=7)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.9992 | Y_std=0.0000 | Y_norm=14.6524
  Cluster 1: R2=0.9851 | Y_std=0.2833 | Y_norm=13.7997
  Cluster 2: R2=0.9805 | Y_std=0.2810 | Y_norm=13.3788
  Cluster 3: R2=0.9874 | Y_std=0.2899 | Y_norm=13.7289
  Cluster 4: R2=1.0000 | Y_std=0.2068 | Y_norm=14.4184
  Cluster 5: R2=0.9673 | Y_std=0.2464 | Y_norm=13.5544
  Cluster 6: R2=0.9552 | Y_std=0.2328 | Y_norm=12.5605

Token 1:
  Cluster 0: R2=1.0000 | Y_std=0.2038 | Y_norm=13.2850
  Cluster 1: R2=0.9888 | Y_std=0.2645 | Y_norm=13.0516
  Cluster 2: R2=0.9677 | Y_std=0.0000 | Y_norm=12.9695
  Cluster 3: R2=0.9870 | Y_std=0.2688 | Y_norm=13.1252
  Cluster 4: R2=0.9903 | Y_std=0.3012 | Y_norm=13.9773
  Cluster 5: R2=0.9638 | Y_std=0.2637 | Y_norm=13.7692
  Cluster 6: R2=0.9637 | Y_std=0.2404 | Y_norm=12.4067

Token 2:
  Cluster 0: R2=0.9855 | Y_std=0.2792 | Y_norm=13.0799
  Cluster 1: R2=0.9838 | Y_std=0.2437 | Y_norm=12.2645
  Cluster 2: R2=0.9840 | Y_std=0.2922 | Y_norm=13.1980
  Clus

In [18]:
inspect_variance(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10,n_clusters=5)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.9609 | Y_std=0.1965 | Y_norm=12.5761
  Cluster 1: R2=0.9869 | Y_std=0.3172 | Y_norm=13.6696
  Cluster 2: R2=1.0000 | Y_std=0.2531 | Y_norm=14.5618
  Cluster 3: R2=0.9818 | Y_std=0.2914 | Y_norm=13.3801
  Cluster 4: R2=0.9866 | Y_std=0.3058 | Y_norm=13.8296

Token 1:
  Cluster 0: R2=0.9643 | Y_std=0.2115 | Y_norm=12.4563
  Cluster 1: R2=0.9867 | Y_std=0.2727 | Y_norm=13.2021
  Cluster 2: R2=1.0000 | Y_std=0.2605 | Y_norm=13.1953
  Cluster 3: R2=0.9859 | Y_std=0.3006 | Y_norm=13.2518
  Cluster 4: R2=0.9905 | Y_std=0.3401 | Y_norm=13.9356

Token 2:
  Cluster 0: R2=0.9797 | Y_std=0.2608 | Y_norm=12.2855
  Cluster 1: R2=0.9854 | Y_std=0.2840 | Y_norm=13.0214
  Cluster 2: R2=0.9640 | Y_std=0.2495 | Y_norm=13.2656
  Cluster 3: R2=0.9796 | Y_std=0.3161 | Y_norm=12.9355
  Cluster 4: R2=1.0000 | Y_std=0.2543 | Y_norm=12.9621


In [ ]:
def analyze_linear_mechanism_full(model, process, regression_mid, regression_post, input_seq, layer_idx=0):
    if input_seq.dim() == 1: input_seq = input_seq.unsqueeze(0)

    cache_names = [
        f"blocks.{layer_idx}.hook_resid_mid", 
        f"blocks.{layer_idx}.hook_resid_post",
        f"blocks.{layer_idx}.hook_mlp_out"
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_seq, names_filter=cache_names)

    resid_mid = cache[cache_names[0]][0].cpu().numpy()
    resid_post = cache[cache_names[1]][0].cpu().numpy()
    mlp_out = cache[cache_names[2]][0].cpu().numpy()
    tokens = input_seq[0].cpu().numpy()
    
    # 2. Project Both to Belief Space
    pred_con = regression_mid.predict(resid_mid)   # Model's Constrained Belief
    pred_lin = regression_post.predict(resid_post)  # Model's Final Belief (after MLP)
    
    #theory
    T_raw, _ = process._create_hmm() 
    T_decay = T_raw.sum(axis=0) 
    
    if hasattr(process, "_create_norm_matrix"):
        S_norm = process._create_norm_matrix()
    else:
        S_norm = np.zeros_like(T_raw)
        for z in range(3):
            S_norm[z] = T_raw[z] / T_raw[z].sum(axis=1, keepdims=True)

    pi = np.array([1/3, 1/3, 1/3])
    curr_lin_belief = pi.copy()
    
    def fmt(v): return f"[{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}]"
    
    print(f"\n{'Pos':<3} {'Tok':<3} {'Err(In)':<8} {'Err(Out)':<8} {'MLP_Norm':<8} {'Theory(Con)':<20} {'Pred(Con)':<20} {'Theory(Lin)':<20} {'Pred(Lin)':<20}")
    print("-" * 140)

    for d, token in enumerate(tokens):
        token_val = int(token)
        pos = d + 1
        
        # Theory Linear
        S_z = S_norm[token_val]
        curr_lin_belief = curr_lin_belief @ S_z
        
        # Theory Constrained
        curr_con_belief = pi.copy()
        for i in range(pos):
            z_i = int(tokens[i])
            S_zi = S_norm[z_i]
            power = (d - i)
            T_pow = np.linalg.matrix_power(T_decay, power)
            term = (pi @ S_zi @ T_pow) - pi
            curr_con_belief += term
            

        err_in = np.linalg.norm(pred_con[d] - curr_con_belief)
        
        err_out = np.linalg.norm(pred_lin[d] - curr_lin_belief)
        
        mlp_norm = np.linalg.norm(mlp_out[d])
        
        print(f"{d:<3} {token_val:<3} {err_in:<8.4f} {err_out:<8.4f} {mlp_norm:<8.2f} {fmt(curr_con_belief):<20} {fmt(pred_con[d]):<20} {fmt(curr_lin_belief):<20} {fmt(pred_lin[d]):<20}")


In [86]:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)

In [89]:
analyze_linear_mechanism_full(model, process, regression_mid=regression_mid, regression_post=regression_post, input_seq=input_seq)


Pos Tok Err(In)  Err(Out) MLP_Norm Theory(Con)          Pred(Con)            Theory(Lin)          Pred(Lin)           
--------------------------------------------------------------------------------------------------------------------------------------------
0   1   0.1267   0.0302   12.97    [0.24 0.52 0.24]     [0.19 0.63 0.18]     [0.24 0.52 0.24]     [0.22 0.55 0.23]    
1   1   0.0072   0.0187   12.74    [0.19 0.63 0.19]     [0.19 0.63 0.18]     [0.19 0.62 0.19]     [0.20 0.61 0.20]    
2   1   0.0577   0.0472   13.11    [0.16 0.68 0.16]     [0.18 0.64 0.18]     [0.16 0.68 0.16]     [0.18 0.64 0.19]    
3   2   0.0781   0.0234   12.54    [0.14 0.43 0.43]     [0.17 0.37 0.46]     [0.18 0.39 0.43]     [0.19 0.40 0.41]    
4   0   0.0391   0.0094   13.95    [0.42 0.29 0.29]     [0.45 0.27 0.28]     [0.44 0.27 0.29]     [0.43 0.28 0.29]    
5   0   0.0298   0.0052   13.14    [0.57 0.22 0.21]     [0.55 0.22 0.23]     [0.58 0.21 0.21]     [0.58 0.21 0.21]    
6   0   0.0781   0.0081  

In [15]:
res = analyze_mlp_math(model, process)

tensor([[2, 0, 1, 1, 0, 1, 0, 1, 2, 2]], device='mps:0')
nothing
nothing
nothing


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def check_embedding_resid_diff(model, process, layer_idx=0, batch_size=128, seq_len=10):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    
    model.to(device)
    
    # Handle process transition to GPU
    if hasattr(process, "_ensure_gpu_tensors"):
        process._ensure_gpu_tensors(device)
    elif hasattr(process, "transition_matrix"):
         if isinstance(process.transition_matrix, np.ndarray):
             process.transition_matrix = torch.tensor(process.transition_matrix, dtype=torch.float32)
         process.transition_matrix = process.transition_matrix.to(device)
         if hasattr(process, "T") and isinstance(process.T, torch.Tensor):
             process.T = process.T.to(device)

    # Generate Batch
    if hasattr(process, "generate_batch_gpu"):
        inputs = process.generate_batch_gpu(batch_size=batch_size, seq_len=seq_len, device=device)
    else:
        raise ValueError("Process does not support generate_batch_gpu")

    # Run Model with Cache
    cache_names = [
        f"blocks.{layer_idx}.hook_resid_mid",    # 0
        f"blocks.{layer_idx}.ln1.hook_normalized", # 1
        f"blocks.{layer_idx}.hook_resid_pre",    # 2
        f"blocks.{layer_idx}.hook_attn_out",     # 3
        f"blocks.{layer_idx}.hook_mlp_out",  
        f"blocks.{layer_idx}.ln"
    ]
    
    with torch.no_grad():
        _, cache = model.run_with_cache(inputs, names_filter=cache_names)
    
    r_mid = cache[cache_names[0]]
    ln1_norm = cache[cache_names[1]]
    r_pre = cache[cache_names[2]]
    attn_out = cache[cache_names[3]]
    mlp_out = cache[cache_names[4]]

    # --- Helpers ---
    def get_stats(t1, t2):
        sim = F.cosine_similarity(t1, t2, dim=-1)
        return sim.mean(dim=0), sim.std(dim=0)

    def get_angle_deg(v1, v2):
        cos = F.cosine_similarity(v1, v2, dim=-1)
        cos = torch.clamp(cos, -1.0 + 1e-6, 1.0 - 1e-6)
        return torch.rad2deg(torch.acos(cos))

    # --- 1. Cosine Similarities ---
    # Define pairs: (Name, Tensor1, Tensor2)
    pairs = [
        ("M-A", r_mid, attn_out),
        ("M-P", r_mid, r_pre),
        ("M-MLP", r_mid, mlp_out),
        ("A-MLP", attn_out, mlp_out),
        ("MLP-P", mlp_out, r_pre),
        ("LN-P", ln1_norm, r_pre),
        ("A-P", attn_out, r_pre)
    ]
    
    # Pre-calculate stats for all pairs
    pair_stats = []
    for name, t1, t2 in pairs:
        m, s = get_stats(t1, t2)
        pair_stats.append((name, m, s))

    print("A=attn_out, M=resid_mid, P=resid_pre, MLP=mlp_out, LN=ln1_norm")    

    print(f"\n{'='*40} Layer {layer_idx}: Cosine Similarities {'='*40}")
    # Dynamic header
    header = f"{'Pos':<4} | " + " | ".join([f"{p[0]:<11}" for p in pair_stats])
    print(header)
    print("-" * len(header))

    for i in range(seq_len):
        row_str = f"{i:<4} | "
        for _, means, stds in pair_stats:
            # Format: 0.99(0.01)
            val = f"{means[i]:.2f}({stds[i]:.2f})"
            row_str += f"{val:<11} | "
        print(row_str[:-3]) # trim last pipe

    # --- 2. Planarity & MLP Orientation ---
    # Hypothesis: Angle(A, M) + Angle(M, P) = Angle(A, P) (A, M, P are collinear-ish in plane)
    ang_AM = get_angle_deg(attn_out, r_mid)
    ang_MP = get_angle_deg(r_mid, r_pre)
    ang_AP = get_angle_deg(attn_out, r_pre)
    
    # MLP angles
    ang_MLP_A = get_angle_deg(mlp_out, attn_out)
    ang_MLP_P = get_angle_deg(mlp_out, r_pre)
    ang_MLP_M = get_angle_deg(mlp_out, r_mid)

    print(f"\n{'='*40} Geometric Structure (Angles in Degrees) {'='*40}")
    print("Planarity Check: A-M + M-P ≈ A-P")
    print("MLP Range Check: Is MLP ~same angle to others?")
    
    header_geo = f"{'Pos':<4} | {'A-M':<6} {'M-P':<6} {'Sum':<6} {'A-P':<6} {'Diff':<6} | {'MLP-A':<6} {'MLP-P':<6} {'MLP-M':<6}"
    print(header_geo)
    print("-" * len(header_geo))

    for i in range(seq_len):
        # Planarity stats
        am = ang_AM[:, i].mean().item()
        mp = ang_MP[:, i].mean().item()
        ap = ang_AP[:, i].mean().item()
        s_ang = am + mp
        diff = abs(s_ang - ap)
        
        # MLP stats
        ma = ang_MLP_A[:, i].mean().item()
        m_p = ang_MLP_P[:, i].mean().item()
        mm = ang_MLP_M[:, i].mean().item()

        print(f"{i:<4} | {am:<6.1f} {mp:<6.1f} {s_ang:<6.1f} {ap:<6.1f} {diff:<6.1f} | {ma:<6.1f} {m_p:<6.1f} {mm:<6.1f}")

    # --- 3. Embedding Reconstruction Check ---
    embeds = model.W_E[inputs]
    pos_embeds = model.W_pos[:seq_len]
    raw_input = embeds + pos_embeds.unsqueeze(0)
    
    sim_raw, std_raw = get_stats(r_mid, raw_input)
    diff_norms = (r_mid - raw_input).norm(dim=-1).mean(dim=0)

    print(f"\n{'='*40} Resid_Mid vs Raw Embedding {'='*40}")
    print(f"{'Pos':<4} | {'Cos Sim (Std)':<15} | {'Diff Norm':<10}")
    print("-" * 35)
    for i in range(seq_len):
        print(f"{i:<4} | {sim_raw[i]:.4f} ({std_raw[i]:.2f})   | {diff_norms[i]:.4f}")


In [85]:
check_embedding_resid_diff(model, process, layer_idx=0, batch_size=256, seq_len=10)

Moving model to device:  mps
A=attn_out, M=resid_mid, P=resid_pre, MLP=mlp_out, LN=ln1_norm

======================================== Layer 0: Cosine Similarities ========================================
Pos  | M-A         | M-P         | M-MLP       | A-MLP       | MLP-P       | LN-P        | A-P        
------------------------------------------------------------------------------------------------------
0    | 0.94(0.01)  | 0.38(0.06)  | 0.16(0.06)  | 0.13(0.08)  | 0.10(0.06)  | 0.99(0.00)  | 0.05(0.03) 
1    | 0.91(0.02)  | 0.52(0.07)  | 0.24(0.06)  | 0.18(0.08)  | 0.21(0.03)  | 1.00(0.00)  | 0.12(0.07) 
2    | 0.90(0.02)  | 0.48(0.07)  | 0.27(0.03)  | 0.24(0.06)  | 0.14(0.06)  | 0.99(0.01)  | 0.07(0.05) 
3    | 0.92(0.02)  | 0.52(0.04)  | 0.24(0.04)  | 0.26(0.05)  | 0.05(0.05)  | 1.00(0.00)  | 0.15(0.03) 
4    | 0.91(0.02)  | 0.40(0.07)  | 0.20(0.04)  | 0.26(0.05)  | -0.09(0.05) | 1.00(0.00)  | -0.01(0.05)
5    | 0.91(0.02)  | 0.47(0.06)  | 0.28(0.03)  | 0.26(0.04)  | 0.11(0.07)  

In [90]:
for name in model.hook_dict.keys():
    print(name)

hook_embed
hook_pos_embed
blocks.0.ln1.hook_scale
blocks.0.ln1.hook_normalized
blocks.0.ln2.hook_scale
blocks.0.ln2.hook_normalized
blocks.0.attn.hook_k
blocks.0.attn.hook_q
blocks.0.attn.hook_v
blocks.0.attn.hook_z
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.hook_result
blocks.0.mlp.hook_pre
blocks.0.mlp.hook_post
blocks.0.hook_attn_in
blocks.0.hook_q_input
blocks.0.hook_k_input
blocks.0.hook_v_input
blocks.0.hook_mlp_in
blocks.0.hook_attn_out
blocks.0.hook_mlp_out
blocks.0.hook_resid_pre
blocks.0.hook_resid_mid
blocks.0.hook_resid_post
ln_final.hook_scale
ln_final.hook_normalized


In [93]:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)

with torch.no_grad():
    # Run model and capture everything (names_filter=None by default)
    _, cache = model.run_with_cache(input_seq)

# Print all hooks in order
for name in cache.keys():
    print(name)

hook_embed
hook_pos_embed
blocks.0.hook_resid_pre
blocks.0.ln1.hook_scale
blocks.0.ln1.hook_normalized
blocks.0.attn.hook_q
blocks.0.attn.hook_k
blocks.0.attn.hook_v
blocks.0.attn.hook_attn_scores
blocks.0.attn.hook_pattern
blocks.0.attn.hook_z
blocks.0.hook_attn_out
blocks.0.hook_resid_mid
blocks.0.ln2.hook_scale
blocks.0.ln2.hook_normalized
blocks.0.mlp.hook_pre
blocks.0.mlp.hook_post
blocks.0.hook_mlp_out
blocks.0.hook_resid_post
ln_final.hook_scale
ln_final.hook_normalized
